In [1]:
import time
import psutil
import os
import threading
import traceback
import random
import copy
from pynvml import *
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

In [2]:
class GateType(Enum):
    X = auto()
    Y = auto()
    Z = auto()
    H = auto()

GATE_UNITARIES: Dict[GateType, List[List[complex]]]= {
    GateType.X: [[0, 1],
                [1, 0]],
    GateType.Y: [[0, -1j],
                [1j, 0]],
    GateType.Z: [[1, 0],
                [0, -1]],
    GateType.H: [[1/2**0.5, 1/2**0.5],
                [1/2**0.5, -1/2**0.5]],
}

@dataclass
class SingleQubitGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int

@dataclass
class ControlledGate:
    gate: GateType
    a: complex
    b: complex
    c: complex
    d: complex
    target: int
    control: List[int]

GateOperation = SingleQubitGate | ControlledGate

def customize(gate: GateType, target: int, control: List[int] | None = None) -> GateOperation:
    matrix = GATE_UNITARIES[gate]
    a, b = matrix[0]
    c, d = matrix[1]
    if control == None:
        return SingleQubitGate(gate, a, b, c, d, target)
    else:
        return ControlledGate(gate, a, b, c, d, target, control)


def randomizer(option: Optional[GateOperation], threshold=0) -> List[GateOperation]:
    operations: List[GateOperation] = []

    for _ in range(NUM_OPERATIONS):
        gate = random.choice(list(GateType))
        matrix = GATE_UNITARIES[gate]
        a, b = matrix[0]
        c, d = matrix[1]
        
        if option == SingleQubitGate or random.random() < threshold:
            target = random.randint(0, NUM_QUBITS - 1)
            operations.append(SingleQubitGate(gate, a, b, c, d, target))
        elif option == ControlledGate or random.random() >= threshold:
            control, target = random.sample(range(NUM_QUBITS), 2)
            operations.append(ControlledGate(gate, a, b, c, d, target, [control]))
    
    return operations

def initialization(L, R) -> Tuple[List[List[complex]], List[List[complex]], List[List[int]]]:    
    if 1 < INIT_MATRIX_SIZE:
        L[1, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            L[row, col] = 1

    if 2 < INIT_MATRIX_SIZE:
        R[2, 0] = 1  

    for col in range(1, INIT_MATRIX_SIZE - 2):  
        if col % 2 == 1:  
            row = col + 3
        else:  
            row = col + 2
        if row < INIT_MATRIX_SIZE:
            R[row, col] = 1

    Mapping = []
    Mapping.append([0])

    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*i + 1])
    Mapping.append([2*i + 2])

    return L, R, Mapping

def initialization_sparse(INIT_MATRIX_SIZE, NUM_QUBITS, TYPE):
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
        
    row_L, col_L, data_L = [], [], []
    row_R, col_R, data_R = [], [], []

    if 1 < INIT_MATRIX_SIZE:
        row_L.append(1)
        col_L.append(0)
        data_L.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 2
        if row < INIT_MATRIX_SIZE:
            row_L.append(row)
            col_L.append(col)
            data_L.append(1)

    if 2 < INIT_MATRIX_SIZE:
        row_R.append(2)
        col_R.append(0)
        data_R.append(1)

    for col in range(1, INIT_MATRIX_SIZE - 2):
        row = col + 3 if col % 2 == 1 else col + 2
        if row < INIT_MATRIX_SIZE:
            row_R.append(row)
            col_R.append(col)
            data_R.append(1)

    L = sparse.csr_matrix((cp.array(data_L, dtype=cp.complex128), (cp.array(row_L), cp.array(col_L))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    R = sparse.csr_matrix((cp.array(data_R, dtype=cp.complex128), (cp.array(row_R), cp.array(col_R))),
                          shape=(INIT_MATRIX_SIZE, INIT_MATRIX_SIZE))

    Mapping = [[0]]
    for i in range(1, NUM_QUBITS):
        Mapping.append([2*i - 1, 2*i])
    Mapping.append([2*NUM_QUBITS - 1])
    Mapping.append([2*NUM_QUBITS])

    return L, R, Mapping
def _view_operations(operations: List[GateOperation]):
    for op in operations:
        print(op.gate.name, op.target)

In [3]:
import re
from pathlib import Path

def extract_qubit_index(s: str) -> int:
    match = re.search(r'\[(\d+)\]', s)
    if match:
        return int(match.group(1))
    raise ValueError(f"Invalid qubit format: {s}")

def parse_qasm_to_custom(path: Path) -> Tuple[int, List[GateOperation]]:
    gate_str_to_enum = {
        'x': GateType.X,
        'y': GateType.Y,
        'z': GateType.Z,
        'h': GateType.H,
    }

    num_qubits = None
    gate_list = []

    with open(path, 'r') as f:
        for line in f:
            line = line.strip().rstrip(';')
            if not line or line.startswith('//') or line.startswith('OPENQASM') or line.startswith('include'):
                continue

            # Parse number of qubits from qreg
            if line.startswith('qreg'):
                match = re.search(r'qreg\s+(\w+)\[(\d+)\]', line)
                if match:
                    num_qubits = int(match.group(2))
                continue

            # Match: gate_name qubit_list
            match = re.match(r'(\w+)\s+(.*)', line)
            if not match:
                print(f"Malformed line: {line}")
                continue

            gate_token, qubit_arg = match.groups()
            qubit_strs = [s.strip() for s in qubit_arg.split(',')]

            # Multi-controlled gate (e.g. cx, ccx, cccx, etc.)
            control_prefix = re.match(r'c+', gate_token)
            if control_prefix and len(gate_token) > len(control_prefix.group()):
                num_controls = len(control_prefix.group())
                base_gate_str = gate_token[num_controls:]

                if base_gate_str not in gate_str_to_enum:
                    print(f"Unsupported multi-controlled gate: {gate_token}")
                    continue

                if len(qubit_strs) != num_controls + 1:
                    print(f"Control/target mismatch in: {line}")
                    continue

                control_indices = [extract_qubit_index(q) for q in qubit_strs[:-1]]
                target_index = extract_qubit_index(qubit_strs[-1])
                gate_enum = gate_str_to_enum[base_gate_str]

                gate_list.append(customize(gate_enum, target_index, control_indices))
                continue

            # Single-qubit gate
            if gate_token in gate_str_to_enum and len(qubit_strs) == 1:
                target = extract_qubit_index(qubit_strs[0])
                gate_enum = gate_str_to_enum[gate_token]
                gate_list.append(customize(gate_enum, target))
                continue

            print(f"Skipping unsupported or malformed line: {line}")

    if num_qubits is None:
        raise ValueError("No qreg declaration found in QASM file.")

    return num_qubits, gate_list


In [4]:
# NUM_QUBITS = 2
# NUM_OPERATIONS = 20
# INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1

# operations = randomizer(option=ControlledGate)

# operations : List[GateOperation] = []
# operations.append(customize(GateType.X, 0))
# operations.append(customize(GateType.H, 0))
# # operations.append(customize(GateType.H, 1))
# operations.append(customize(GateType.X, 1, [0]))
# # operations.append(customize(GateType.H, 0))
# operations.append(customize(GateType.X, 1, [0]))
# operations.append(customize(GateType.X, 2, [1]))

# operations.append(customize(GateType.Z, 2))

# _view_operations(operations)

In [5]:
def compute_result(L, R, seq: List[int], leaf: List[int], TYPE) -> complex:
    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
        
    A = cp.zeros((1, L.shape[0]))

    A[0, leaf] = 1

    B = cp.zeros((L.shape[0], 1))
    B[0, 0] = 1

    dirs = B
    for dir in seq:
        if dir == 0:
            dirs = L @ dirs
        else:
            dirs = R @ dirs
        
    result = A @ dirs

    return result.item()

class SimulationAllPath:
    def __init__(self, L, R, leaf, NUM_QUBITS, TYPE):
        self.L = L
        self.R = R
        self.LL = self.L@self.L
        self.RR = self.R@self.R
        self.LR = self.L@self.R
        self.RL = self.R@self.L
        self.leaf = leaf
        self.NUM_QUBITS = NUM_QUBITS
        self.TYPE = TYPE
        self.sim_result = {}

    def recursion(self, seq: List[int], dirs, A, start_time):
        if time.time() - start_time > 60:
            raise TimeoutError("Simulation Timeout: 60s.")
        if len(seq) == self.NUM_QUBITS:
            result = A @ dirs
            val = result.toarray()[0, 0]
            if abs(val) > 1e-10:
                self.sim_result[tuple(seq)] = val
            return
        
        if self.NUM_QUBITS <= 10 or len(seq) < 2:
            for dir in [0, 1]:
                next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                self.recursion(seq + [dir], next_dirs, A, start_time)
        else:
            for d1 in [0, 1]:
                for d2 in [0, 1]:
                    if d1 == 0 and d2 == 0:
                        next_dirs = self.LL @ dirs
                    elif d1 == 0 and d2 == 1:
                        next_dirs = self.RL @ dirs
                    elif d1 == 1 and d2 == 0:
                        next_dirs = self.LR @ dirs
                    elif d1 == 1 and d2 == 1:
                        next_dirs = self.RR @ dirs
                    
                    self.recursion(seq + [d1, d2], next_dirs, A, start_time)

    def simulate_non_recursive(self, A, B, start_time):
        stack = [([], B)]  # (sequence_so_far, current_dirs)

        while stack:
            if time.time() - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            if self.NUM_QUBITS <= 10 or len(seq) == self.NUM_QUBITS - 1:
                for dir in [0, 1]:
                    next_dirs = self.L @ dirs if dir == 0 else self.R @ dirs
                    stack.append((seq + [dir], next_dirs))
            else:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        if d1 == 0 and d2 == 0:
                            next_dirs = self.LL @ dirs
                        elif d1 == 0 and d2 == 1:
                            next_dirs = self.RL @ dirs
                        elif d1 == 1 and d2 == 0:
                            next_dirs = self.LR @ dirs
                        else:  # d1 == 1 and d2 == 1
                            next_dirs = self.RR @ dirs

                        stack.append((seq + [d1, d2], next_dirs))

    def build_4bit_matrix_map(self):
        from itertools import product

        mat_map = {}
        for bits in product([0, 1], repeat=4):
            mat = self.L if bits[0] == 0 else self.R
            for b in bits[1:]:
                mat = (self.L if b == 0 else self.R) @ mat
            mat_map[bits] = mat
        return mat_map
    
    def simulate_non_recursive_4bit(self, A, B, start_time):
        matrix4_map = self.build_4bit_matrix_map()
        stack = [([], B)]

        while stack:
            now_time = time.time()
            if now_time - start_time > 60:
                raise TimeoutError("Simulation Timeout: 60s.")

            seq, dirs = stack.pop()

            if len(seq) == self.NUM_QUBITS:
                result = A @ dirs
                val = result.toarray()[0, 0]
                if abs(val) > 1e-10:
                    self.sim_result[tuple(seq)] = val
                continue

            remaining = self.NUM_QUBITS - len(seq)
            if remaining >= 4:
                for bits4, mat in matrix4_map.items():
                    next_seq = seq + list(bits4)
                    next_dirs = mat @ dirs
                    stack.append((next_seq, next_dirs))
            elif remaining >= 2:
                for d1 in [0, 1]:
                    for d2 in [0, 1]:
                        next_dirs = (
                            self.LL @ dirs if d1 == 0 and d2 == 0 else
                            self.RL @ dirs if d1 == 0 and d2 == 1 else
                            self.LR @ dirs if d1 == 1 and d2 == 0 else
                            self.RR @ dirs
                        )
                        stack.append((seq + [d1, d2], next_dirs))
            else:
                for d in [0, 1]:
                    next_dirs = self.L @ dirs if d == 0 else self.R @ dirs
                    stack.append((seq + [d], next_dirs))
            
            print(time.time() - now_time, "seconds elapsed for current iteration:", seq)


    def simulation(self):
        if self.TYPE == 'numpy':
            import numpy as cp
            from scipy import sparse
        elif self.TYPE == 'cupy':
            import cupy as cp
            from cupyx.scipy import sparse

        A = cp.zeros((1, self.L.shape[0]))
        A[0, self.leaf] = 1
        A = sparse.csr_matrix(A, dtype=cp.complex128)

        B = cp.zeros((self.L.shape[0], 1))
        B[0, 0] = 1
        B = sparse.csr_matrix(B, dtype=cp.complex128)

        start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)
        self.simulate_non_recursive_4bit(A, B, start_simulation)


    def __iter__(self):
        ...
        # A = cp.zeros((1, self.L.shape[0]))
        # A[0, self.leaf] = 1
        # A = sparse.csc_matrix(A, dtype=cp.complex128)

        # B = cp.zeros((self.L.shape[0], 1))
        # B[0, 0] = 1
        # B = sparse.csc_matrix(B, dtype=cp.complex128)

        # start_simulation = time.time()

        # self.recursion([], B, A, start_simulation)

        # for i in range(2**self.NUM_QUBITS):
        #     if time.time() - start_simulation > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")

        #     seq = [(i >> j) & 1 for j in reversed(range(self.NUM_QUBITS))]
        #     dirs = B
        #     if self.NUM_QUBITS <= 10:
        #         for dir in seq:
        #             dirs = self.L @ dirs if dir == 0 else self.R @ dirs
        #     else:
        #         for d1, d2 in zip(seq[::2], seq[1::2]):
        #             if d1 == 0 and d2 == 0:
        #                 dirs = self.LL @ dirs
        #             elif d1 == 0 and d2 == 1:
        #                 dirs = self.RL @ dirs
        #             elif d1 == 1 and d2 == 0:
        #                 dirs = self.LR @ dirs
        #             elif d1 == 1 and d2 == 1:
        #                 dirs = self.RR @ dirs
            
        #         if self.NUM_QUBITS % 2 == 1:
        #             d = seq[-1]
        #             dirs = self.L @ dirs if d == 0 else self.R @ dirs

        #     result = A @ dirs
            # val = result.toarray()[0, 0]
            # val = result.item()
            # print("L: ", self.L)
            # print("R: ", self.R)
            # print("Leaf: ", self.leaf)
            # print(f"Path: {seq}, Value: {val}")
            # if abs(val) > 1e-10:
            #     self.sim_result[tuple(seq)] = val
            # yield val, seq

    def results(self):
        return self.sim_result

def simulation_all_path(L, R, leaf, NUM_QUBITS, TYPE):
    sim = SimulationAllPath(L, R, leaf, NUM_QUBITS, TYPE)
    sim.simulation()
    # for val, seq in sim:
    #     pass  # Iterate through all paths to compute results
    return sim.results()



In [6]:
def dd_sparse(operations: List[GateOperation], MATRIX_SIZE: int, NUM_QUBITS: int, TYPE: str):
    L, R, Mapping = initialization_sparse(MATRIX_SIZE, NUM_QUBITS, TYPE)
    # print("Init Mapping: ", Mapping)
    total_start = time.time()

    if TYPE == 'numpy':
        import numpy as cp
        from scipy import sparse
    elif TYPE == 'cupy':
        import cupy as cp
        from cupyx.scipy import sparse
    
    for op in operations:
        if time.time() - total_start > 60:
            raise TimeoutError("Timeout: 60s.")
        
        MATRIX_SIZE = L.shape[0]
        # print("MATRIX_SIZE: ", MATRIX_SIZE)
        # print("L: ", L)
        # print("R: ", R)
        L_U, R_U = copy.deepcopy(L), copy.deepcopy(R)
        
        A = cp.ones(MATRIX_SIZE, dtype=complex)
        A[Mapping[op.target]] = op.a
        B = cp.zeros(MATRIX_SIZE, dtype=complex)
        B[Mapping[op.target]] = op.b
        C = cp.zeros(MATRIX_SIZE, dtype=complex)
        C[Mapping[op.target]] = op.c
        D = cp.ones(MATRIX_SIZE, dtype=complex)
        D[Mapping[op.target]] = op.d

        A = sparse.diags(A, offsets=0, format='csr')
        B = sparse.diags(B, offsets=0, format='csr')
        C = sparse.diags(C, offsets=0, format='csr')
        D = sparse.diags(D, offsets=0, format='csr')
    

        L_U = L @ A + R @ B
        R_U = L @ C + R @ D

        if isinstance(op, SingleQubitGate):
            L, R = L_U, R_U
            # print("Mapping: ", Mapping)
        
        elif isinstance(op, ControlledGate):
            MATRIX_SIZE_OLD = MATRIX_SIZE
            for control in reversed(op.control):
                keep = cp.zeros(MATRIX_SIZE_OLD, dtype=int)
                keep_complement = cp.ones(MATRIX_SIZE_OLD, dtype=int)
                keep[Mapping[control]] = 1
                keep_complement[Mapping[control]] = 0
                D = sparse.diags(keep, offsets=0, format='csr')
                D_complement = sparse.diags(keep_complement, offsets=0, format='csr')
                R_masked = R @ D_complement
                R_U_masked = R_U[:, :MATRIX_SIZE_OLD] @ D

                L_U = sparse.bmat([[L, None],
                    [None, L_U]], format='csr')
                
                # print("Mapping: ", Mapping[control])
                
                # print("D: ", D)
                # print("D_complement: ", D_complement)
                # print("R: ", R)
                # print("R_U: ", R_U)
                # print("R_U[]: ", R_U[:, :MATRIX_SIZE_OLD])
                # print("R_masked: ", R_masked)
                # print("R_U_masked: ", R_U_masked)

                
                # if TYPE == 'cupy':
                #     print("R Masked Type: ", type(R_masked))
                #     print("R U Masked Type: ", type(R_U_masked))
                #     print("R U Type: ", type(R_U))
                #     print("R Masked shape: ", R_masked.shape)
                #     print("R U Masked shape: ", R_U_masked.shape)
                #     print("R_U shape: ", R_U.shape)
                    # print("R_masked max row/col:", R_masked.nonzero())
                    # print("R_U_masked max row/col:", R_U_masked.nonzero())
                    # print("R_U max row/col:", R_U.nonzero())

                R_U = sparse.bmat([[R_masked, None],
                    [R_U_masked, R_U]], format='csr')
                
                # print("L_U: ", L_U)
                # print("R_U: ", R_U)
                
                MATRIX_SIZE += MATRIX_SIZE_OLD

            L, R = L_U, R_U
            # print("start traverse")
            traverse_start = time.time()

            L_B = L.astype(bool).astype(float)
            R_B = R.astype(bool).astype(float)

            LR = L_B + R_B
            LR = LR.astype(bool)
            LR = LR.astype(float)

            traverse = sparse.csr_matrix((MATRIX_SIZE, 1), dtype=bool)
            traverse[0, 0] = True
            prev = -1
            while traverse.nnz != prev:
                prev = traverse.nnz
                traverse = traverse.astype(float)
                traverse = (traverse + LR @ traverse)
                traverse = traverse.astype(bool)

            traverse = traverse.astype(float)
            traverse = set(traverse.toarray().ravel().nonzero()[0].tolist())

            # print("Traverse: ", traverse)
            # print("end traverse, time spent:", time.time() - traverse_start)

            # print("start mapping")
            mapping_start = time.time()
            Mapping_U = [
                sorted(
                    [n + MATRIX_SIZE_OLD * i
                    for i in range(1, MATRIX_SIZE // MATRIX_SIZE_OLD + 1)
                    for n in mapping
                    if n + MATRIX_SIZE_OLD * i in traverse] +
                    [n for n in mapping if n in traverse]
                )
                for mapping in Mapping
            ]
            # print("Mapping_U: ", Mapping_U)

            mapping_middle = time.time()
            # print("Mapping middle time spent:", mapping_middle - mapping_start)
            traverse_list = list(traverse)
            traverse_dict = {i: idx for idx, i in enumerate(traverse_list)}
            Mapping = [[traverse_dict[node] for node in mapping_U] for mapping_U in Mapping_U]
            # print("Mapping_U after dict: ", Mapping_U)
            # Mapping = copy.deepcopy(Mapping_U)
            # print("end mapping, time spent:", time.time() - mapping_middle)  

            # print("start construct")
            construct_start = time.time()


            not_traverse = cp.array(list(set(range(MATRIX_SIZE)) - traverse))

            rows = cp.arange(MATRIX_SIZE)
            rows = rows[~cp.isin(rows, not_traverse)]
            one = cp.ones_like(rows, dtype=cp.int32)


            S = sparse.csr_matrix((one, (cp.arange(len(rows)), rows)),
                                    shape=(len(rows), MATRIX_SIZE), dtype=cp.complex128)
            
            L = S @ L @ S.T
            R = S @ R @ S.T

            # print("end construct, time spent:", time.time() - construct_start)

            MATRIX_SIZE = L.shape[0]

        if not L.shape == R.shape:
            raise ValueError("Size of L and R not equal")

    # print("MATRIX_SIZE: ", MATRIX_SIZE)
    # print("L: ", L)
    # print("R: ", R)
    # print("Mapping: ", Mapping)
    sim_result = compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2], TYPE)
    # sim_result = simulation_all_path(L, R, Mapping[-2], NUM_QUBITS, TYPE)
    # sim_result = 0
    return sim_result

In [7]:
def dd_sparse_numpy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int):
    
    def monitor_cpu_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)  # percent of a core
            mem = process.memory_info().rss / 1024**2  # in MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)
            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

    import numpy as cp
    import scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_cpu_usage, args=(stop_event, result_dict))

        np_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
            # if time.time() - np_start > 60:
            #     raise TimeoutError("Simulation Timeout: 60s.")
        sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'numpy')

        np_end = time.time()

        stop_event.set()
        monitor_thread.join()

        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = None

        print(sim_result)

        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))
    except Exception as e:
        np_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in numpy + scipy:", e)
        result_dict["time_spent"] = np_end - np_start
        result_dict["error"] = str(e)
        traceback.print_exc()

    # print("Time spent: ", np_end - np_start)
    return result_dict


In [8]:
def dd_sparse_cupy(operations: List[GateOperation], INIT_MATRIX_SIZE: int, NUM_QUBITS: int):
    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)

        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    import cupy as cp
    import cupyx.scipy.sparse as sparse
    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))
    
        cp_start = time.time()
        monitor_thread.start()
        # for i in range(1024):
        #     if time.time() - cp_start > 60:
        #         raise TimeoutError("Simulation Timeout: 60s.")
        sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'cupy')
        
        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("|0>*n: ", compute_result(L, R, [0]*NUM_QUBITS, Mapping[-2]))
        # print("|1>*n: ", compute_result(L, R, [1]*NUM_QUBITS, Mapping[-2]))

        device_id = cp.cuda.runtime.getDevice()
        device_props = cp.cuda.runtime.getDeviceProperties(device_id)
        # print(f"Device {device_id} name:", device_props['name'].decode())
        print(sim_result)
        
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = None

    except Exception as e:
        cp_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in cupy computation:", e)
        result_dict["time_spent"] = cp_end - cp_start
        result_dict["error"] = str(e)
        traceback.print_exc()


    # print("Time spent: ", cp_end - cp_start)
    return result_dict


In [9]:
# cudaq.set_target("nvidia")

# kernel = build_kernel_from_operations(NUM_QUBITS, operations)
# result = cudaq.sample(kernel)
# print(result)


In [10]:
def cudaq_sv(path: str, NUM_QUBITS):
    # NUM_QUBITS = min(NUM_QUBITS, 16)
    NUM_QUBITS = 0
    # output_path = Path(path + 'circuit_cudaq.py')
    # !npx q-convert -i {input_path} -s qasm -o {output_path} -d cudaq

    method_style_gates = {"x", "y", "z", "h", "rx", "ry", "rz", "t", "s", "swap"}

    def parse_qasm_to_cudaq_python(qasm_file: str, output_py_file: str):
        with open(qasm_file, 'r') as f:
            lines = f.readlines()

        cudaq_lines = [
            "import cudaq\n",
            "@cudaq.kernel\n",
            "def circuit():\n"
        ]
        indent = "    "

        for line in lines:
            line = line.strip()
            if not line or line.startswith("OPENQASM") or line.startswith("include") or line.startswith("//"):
                continue

            if match := re.match(r"qreg\s+(\w+)\[(\d+)\];", line):
                regname, size = match.groups()
                cudaq_lines.append(f"{indent}{regname} = cudaq.qvector({size})\n")
                continue

            if match := re.match(r"([a-zA-Z_][a-zA-Z0-9_]*)\s+(.*);", line):
                gate_raw, arglist = match.groups()
                args = [arg.strip() for arg in arglist.split(",") if arg.strip()]

                ctrl_count = len(gate_raw) - len(gate_raw.lstrip("c"))
                base_gate = gate_raw[ctrl_count:]

                if ctrl_count > 0:
                    controls = args[:ctrl_count]
                    targets = args[ctrl_count:]

                    if base_gate in method_style_gates and len(targets) == 1:
                        ctrl_list = f"[{', '.join(controls)}]"
                        cudaq_lines.append(
                            f"{indent}{base_gate}.ctrl({ctrl_list}, {targets[0]})\n"
                        )
                    else:
                        ctrl_tuple = ", ".join(controls)
                        lambda_call = f"{base_gate}({', '.join(targets)})"
                        cudaq_lines.append(
                            f"{indent}cudaq.control(({ctrl_tuple}), lambda: {lambda_call})\n"
                        )
                else:
                    if base_gate in method_style_gates:
                        cudaq_lines.append(f"{indent}{base_gate}({', '.join(args)})\n")
                    else:
                        cudaq_lines.append(f"{indent}# TODO: Unrecognized gate: {base_gate}({', '.join(args)})\n")

        cudaq_lines.append(f"\ncounts = cudaq.sample(circuit, shots_count={2**NUM_QUBITS})\nprint(counts)")
        with open(output_py_file, 'w') as f:
            f.writelines(cudaq_lines)

    
    input_path = Path(path + 'circuit.qasm')
    output_path = Path(path + 'circuit_cudaq_new.py')
    parse_qasm_to_cudaq_python(input_path, output_path)

    try:
        nvmlInit()
        gpu_available = True
        handle = nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_available = False

    def monitor_usage(stop_event, result_dict, interval=0.5):
        pid = os.getpid()
        process = psutil.Process(pid)
        cpu_usages = []
        mem_usages = []
        gpu_usages = []
        gpu_mems = []
        
        while not stop_event.is_set():
            cpu = process.cpu_percent(interval=None)
            mem = process.memory_info().rss / 1024**2  # MB
            cpu_usages.append(cpu)
            mem_usages.append(mem)

            if gpu_available:
                try:
                    util = nvmlDeviceGetUtilizationRates(handle)
                    mem_info = nvmlDeviceGetMemoryInfo(handle)
                    gpu_usages.append(util.gpu)
                    gpu_mems.append(mem_info.used / 1024**2)
                except:
                    gpu_usages.append(0)
                    gpu_mems.append(0)

            time.sleep(interval)
        
        result_dict["avg_cpu"] = sum(cpu_usages)/len(cpu_usages) if cpu_usages else 0.0
        result_dict["peak_mem"] = max(mem_usages) if mem_usages else 0.0

        if gpu_usages:
            result_dict["avg_gpu"] = sum(gpu_usages)/len(gpu_usages)
            result_dict["peak_gpu_mem"] = max(gpu_mems)
        else:
            result_dict["avg_gpu"] = 0.0
            result_dict["peak_gpu_mem"] = 0.0

    try:
        result_dict = {}
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_usage, args=(stop_event, result_dict))

        cudaq_start = time.time()
        monitor_thread.start()

        import subprocess
        # %run -i {output_path}
        result = subprocess.run(
            ['python', output_path],
            capture_output=True,
            text=True,
            check=True  # raises CalledProcessError if returncode != 0
        )
        print("STDOUT: ", result.stdout)

        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict["error"] = None
        # most_probable_result = counts.most_probable()
        # probability = counts.probability(most_probable_result)
        # print("Most probable result: " + most_probable_result)
        # print("Measured with probability " + str(probability), end='\n\n')
        # print("time spent: ", cudaq_end - cudaq_start)
    # except Exception as e:
    except subprocess.CalledProcessError as e:
        cudaq_end = time.time()
        stop_event.set()
        monitor_thread.join()
        # print("Error in cudaq computation:", e)
        result_dict["time_spent"] = cudaq_end - cudaq_start
        result_dict["error"] = str(e.stderr)
        print("Error output:\n", e.stderr)
        traceback.print_exc()
    return result_dict


In [11]:
import pandas as pd
import openpyxl

# benchmarks = {
#               'BV': 1,
#             #   'BVSym': 1, 
#             #   'GHZall': 2, 
#               'GHZzero': 2, 
#               'Grover': 2, 
#             #   'GroverSym': 2, 
#             #   'H2': 1, 
#             #   'HXH': 1, 
#               'MCToffoli': 3, 
#             #   'MOBV_reorder': 1, 
#             #   'MOGrover': 3, 
#             #   'MOGroverSym': 3, 
#               'OEGrover': 2
#               }
benchmarks = {
              'BV': 1,
            #   'BVSym': 1, 
            #   'GHZall': 2, 
            #   'GHZzero': 2, 
            #   'Grover': 2, 
            #   'GroverSym': 2, 
            #   'H2': 1, 
            #   'HXH': 1, 
            #   'MCToffoli': 3, 
            #   'MOBV_reorder': 1, 
            #   'MOGrover': 3, 
            #   'MOGroverSym': 3, 
            #   'OEGrover': 2
              }

benchmark_data = {}

for benchmark in benchmarks:

    methods = ['NumPy + PySparse', 'CuPy + cuSparse', 'CudaQ (State Vector)']
    subfields = ['Time (s)', 'CPU Memory Usage (MB)', 'GPU Memory Usage (MB)', 'Error']

    columns = [('Qubits', '')]
    rows = []

    for method in methods:
        for sub in subfields:
            if method == 'NumPy + PySparse' and sub == 'GPU Memory Usage (MB)':
                continue
            columns.append((method, sub))
    
    multi_columns = pd.MultiIndex.from_tuples(columns)

    flag = (True, True, True)
    
    while (flag[0] or flag[1] or flag[2]) and benchmarks[benchmark] <= 100:

        if benchmark == 'Grover' and benchmarks[benchmark] == 22:
            break

        if benchmark == 'GroverSym' and benchmarks[benchmark] == 22:
            break

        if benchmark == 'MOGrover' and benchmarks[benchmark] == 12:
            break

        if benchmark == 'MOGroverSym' and benchmarks[benchmark] == 12:
            break
        

        i = str(benchmarks[benchmark])
        if int(i) < 10:
            i = '0' + i
        if benchmark in ['GHZall', 'GHZzero', 'H2'] and int(i) < 100:
            i = '0' + i

        path = './AutoQ/benchmarks/all/' + benchmark + f'/{i}/'
        input_path = Path(path + 'circuit.qasm')
        NUM_QUBITS, operations = parse_qasm_to_custom(input_path)

        INIT_MATRIX_SIZE = 2 * NUM_QUBITS + 1
        print(f"Qubit count: {NUM_QUBITS}")
        # print("Operations:", operations)

        if flag[0]:
            np_result = dd_sparse_numpy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
        if flag[1]:
            cp_result = dd_sparse_cupy(operations, INIT_MATRIX_SIZE, NUM_QUBITS)
        if flag[2]:
            cudaq_result = cudaq_sv(path, NUM_QUBITS)

        rows.append([NUM_QUBITS, 
                     np_result['time_spent'] if flag[0] else '--',
                     np_result['peak_mem'] if flag[0] else '--',
                     np_result['error'] if flag[0] else '--',
                     cp_result['time_spent'] if flag[1] else '--',
                     cp_result['peak_mem'] if flag[1] else '--',
                     cp_result['peak_gpu_mem'] if flag[1] else '--',
                     cp_result['error'] if flag[1] else '--',
                     cudaq_result['time_spent'] if flag[2] else '--',
                     cudaq_result['peak_mem'] if flag[2] else '--',
                     cudaq_result['peak_gpu_mem'] if flag[2] else '--',
                     cudaq_result['error'] if flag[2] else '--'
                     ])
                   

        if not np_result['error'] is None:
            flag = (False, flag[1], flag[2])
        
        if not cp_result['error'] is None:
            flag = (flag[0], False, flag[2])
        
        if not cudaq_result['error'] is None:
            flag = (flag[0], flag[1], False)

        benchmarks[benchmark] += 1

    benchmark_data[benchmark] = rows
        

Qubit count: 2


/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


0j


/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


0j
STDOUT:  { 11:1 }


Qubit count: 3
0j


...

/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


0j
Qubit count: 33
0j
0j
Qubit count: 34
0j
0j
Qubit count: 35
0j
0j
Qubit count: 36
0j
0j
Qubit count: 37
0j
0j
Qubit count: 38
0j


Traceback (most recent call last):
  File "/tmp/ipykernel_151511/4120167527.py", line 57, in dd_sparse_cupy
    sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'cupy')
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_151511/2519192177.py", line 154, in dd_sparse
    R = S @ R @ S.T
        ~~^~~
  File "/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_base.py", line 130, in __matmul__
    return self.__mul__(other)
           ^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/cupyx/scipy/sparse/_csr.py", line 159, in __mul__
    return cusparse.spgemm(self, other)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/Yu-Hsuan Wu/autoq/.venv/lib/python3.12/site-packages/cupyx/cusparse.py", line 2066, in spgemm
    _cusparse.spGEMM_compute(
  File "cupy_backends/cuda/libs/cusparse.pyx", line 5075, in cupy_backends.cuda.libs.cusparse.sp

Qubit count: 39
0j
Qubit count: 40


Traceback (most recent call last):
  File "/tmp/ipykernel_151511/963496910.py", line 31, in dd_sparse_numpy
    sim_result = dd_sparse(operations, INIT_MATRIX_SIZE, NUM_QUBITS, 'numpy')
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_151511/2519192177.py", line 15, in dd_sparse
    raise TimeoutError("Timeout: 60s.")
TimeoutError: Timeout: 60s.


In [12]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

excel_output_path = './benchmark_sim_only1.xlsx'

excel_file = Path(excel_output_path)

def write_sheets(writer, data_dict):
    for benchmark, rows in data_dict.items():
        df = pd.DataFrame(rows, columns=multi_columns)
        df.to_excel(writer, sheet_name=benchmark, index=True)

if not excel_file.exists():
    # Create a new Excel file if it does not exist
    with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
        write_sheets(writer, benchmark_data)
else:
    try:
        # Load the workbook
        book = load_workbook(excel_output_path)
        for benchmark in benchmark_data.keys():
            if benchmark in book.sheetnames:
                # Delete the sheet if it exists
                std = book[benchmark]
                book.remove(std)

        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer.book = book  # attach modified workbook
            write_sheets(writer, benchmark_data)

    except Exception as e:
        print(f"Error opening existing Excel file: {e}")
        print("Recreating the file in write mode.")
        with pd.ExcelWriter(excel_output_path, engine='openpyxl', mode='w') as writer:
            write_sheets(writer, benchmark_data)


In [13]:
print(benchmark_data)

{'BV': [[2, 0.012251615524291992, 119.328125, None, 0.46481990814208984, 321.078125, 316.5625, None, 6.2491395473480225, 621.4609375, 480.2890625, None], [3, 0.009400606155395508, 0.0, None, 0.06683802604675293, 621.7109375, 471.42578125, None, 6.338107347488403, 621.7109375, 471.42578125, None], [4, 0.015499353408813477, 621.7109375, None, 0.10822367668151855, 621.7109375, 471.42578125, None, 6.466290712356567, 621.7109375, 471.42578125, None], [5, 0.01093745231628418, 0.0, None, 0.13013100624084473, 621.9609375, 471.42578125, None, 6.351664304733276, 621.9609375, 471.42578125, None], [6, 0.01703047752380371, 621.9609375, None, 0.18509888648986816, 621.9609375, 471.42578125, None, 6.418133974075317, 621.9609375, 471.42578125, None], [7, 0.020769119262695312, 0.0, None, 0.1814265251159668, 621.9609375, 471.42578125, None, 6.175281047821045, 621.9609375, 684.2890625, None], [8, 0.020506620407104492, 621.9609375, None, 0.2373371124267578, 621.9609375, 471.42578125, None, 6.25472259521484